# Pipeline IFRS 9 – Probabilidad de Default (Freddie Mac)
**Maestría en Minería de Datos – UTN Facultad Regional Paraná**
Autor: Sebastian Emiliano Meier | Director: Mag. Ing. Gustavo Denicolay

---

### Estructura del notebook

| Sección | Contenido |
|---|---|
| 0 | Configuración, imports y helpers |
| 1 | Ingesta de datos crudos (`data_reader.py`) |
| 2 | Paso 1 – Panel longitudinal |
| 3 | Paso 2 – Definición de default IFRS 9 |
| **4** | **Análisis temporal: DPD, ODR y curvas de vintage** |
| **5** | **Chequeos de calidad del panel** |
| 6 | Paso 3 – Feature engineering |
| 7 | Paso 4 – Entrenamiento de modelos |
| 8 | Paso 5 – Calibración Lifetime PD |
| 9 | Paso 6 – Validación regulatoria |
| 10 | Paso 7 – Explicabilidad SHAP |


## 0. Configuración

In [ ]:
%matplotlib inline
import sys, os, warnings
from pathlib import Path

# ── Rutas ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent          # FreddieMAC/
SRC_DIR      = PROJECT_ROOT / "src"
DATA_DIR     = PROJECT_ROOT / "data"
FIG_DIR      = PROJECT_ROOT / "outputs" / "figuras"

sys.path.insert(0, str(PROJECT_ROOT))       # para data_reader
sys.path.insert(0, str(SRC_DIR))            # para config y pasos

warnings.filterwarnings("ignore")
os.chdir(SRC_DIR)                           # los scripts usan rutas relativas desde src/

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

# ── Estilo gráfico ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (12, 4),
    "font.family": "serif",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("Blues_d", 8)

print("Configuración OK")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SRC_DIR      : {SRC_DIR}")


## 1. Ingesta de datos crudos (`data_reader.py`)
Lee los archivos `.txt` por vintage y genera `orig_all.parquet` y `svcg_all.parquet`.
**Omitir esta celda si los parquets ya existen.**


In [ ]:
# ── Ejecutar solo si los parquets no existen ─────────────────────────────────
orig_p = DATA_DIR / "orig_all.parquet"
svcg_p = DATA_DIR / "svcg_all.parquet"

if not orig_p.exists() or not svcg_p.exists():
    os.chdir(PROJECT_ROOT)
    import importlib.util, sys as _sys
    spec = importlib.util.spec_from_file_location("data_reader", PROJECT_ROOT / "data_reader.py")
    dr = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(dr)
    dr.main()
    os.chdir(SRC_DIR)
    print("data_reader completado.")
else:
    print("Parquets ya existen – saltando ingesta.")


In [ ]:
# ── Vista rápida de los parquets ─────────────────────────────────────────────
orig = pd.read_parquet(DATA_DIR / "orig_all.parquet")
svcg = pd.read_parquet(DATA_DIR / "svcg_all.parquet")

print(f"orig_all : {orig.shape[0]:,} filas × {orig.shape[1]} cols")
print(f"svcg_all : {svcg.shape[0]:,} filas × {svcg.shape[1]} cols")
print(f"\nVintages en orig : {sorted(orig['vintage_year'].unique())}")
print(f"Vintages en svcg : {sorted(svcg['vintage_year'].unique())}")
print(f"\nPeríodo svcg : {svcg['monthly_reporting_period'].min()} → {svcg['monthly_reporting_period'].max()}")


In [ ]:
# ── Tipos y nulos en originación ─────────────────────────────────────────────
nulos_orig = (orig.isnull().mean() * 100).round(2).to_frame("% nulos")
nulos_orig["dtype"] = orig.dtypes.astype(str)
display(nulos_orig[nulos_orig["% nulos"] > 0].sort_values("% nulos", ascending=False))


## 2. Paso 1 – Panel longitudinal (`01_carga_datos.py`)
Une originación y performance en un panel `(préstamo × mes)`.


In [ ]:
import importlib, paso_01 as _m
importlib.reload(_m); del _m

import importlib.util as _iu, sys as _sys
spec = _iu.spec_from_file_location("m01", SRC_DIR / "01_carga_datos.py")
m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
panel = m.main()


In [ ]:
# ── Cobertura temporal por vintage ───────────────────────────────────────────
cob = (panel.groupby("vintage_year")["monthly_reporting_period"]
       .agg(inicio="min", fin="max", obs="count")
       .reset_index())
cob["obs"] = cob["obs"].map("{:,}".format)
display(cob)


In [ ]:
# ── Calidad del join: préstamos sin originación ───────────────────────────────
sin_orig = panel["credit_score"].isnull().sum()
total    = len(panel)
print(f"Filas sin datos de originación (credit_score NaN): {sin_orig:,} / {total:,}  ({sin_orig/total:.2%})")


## 3. Paso 2 – Definición de default IFRS 9 (`02_definicion_default.py`)
Genera `dpd_numerico`, `evento_default`, `ifrs9_stage` y la variable objetivo `default_12m`.


In [ ]:
import importlib.util as _iu
spec = _iu.spec_from_file_location("m02", SRC_DIR / "02_definicion_default.py")
m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
panel_def = m.main()


In [ ]:
# ── Distribución de stages ────────────────────────────────────────────────────
stage_dist = (panel_def["ifrs9_stage"]
              .value_counts(normalize=True)
              .rename({1: "Stage 1", 2: "Stage 2", 3: "Stage 3"})
              .mul(100).round(2))
print("Distribución de stages (%):")
print(stage_dist.to_string())


## 4. Análisis temporal: DPD, ODR y curvas de vintage

Estos chequeos usan el panel con default (`panel_con_default.parquet`) para diagnosticar
la calidad del portafolio a lo largo del tiempo.


### 4.1 Composición mensual del portafolio por bucket DPD

In [ ]:
# ── Carga eficiente (solo columnas necesarias) ────────────────────────────────
cols = ["monthly_reporting_period", "dpd_numerico", "ifrs9_stage",
        "default_12m", "vintage_year", "loan_sequence_number"]
pdf = pd.read_parquet(DATA_DIR / "panel_con_default.parquet", columns=cols)

# Bucket DPD
def bucket_dpd(x):
    if pd.isna(x): return "Desconocido"
    if x == 0: return "0 – Al día"
    if x == 1: return "1 – 30 DPD"
    if x == 2: return "2 – 60 DPD"
    if x >= 3: return "3+ – Default"
    return "Desconocido"

pdf["bucket"] = pdf["dpd_numerico"].apply(bucket_dpd)
print(f"Panel cargado: {len(pdf):,} filas | {pdf['loan_sequence_number'].nunique():,} préstamos únicos")


In [ ]:
# ── % mensual por bucket ─────────────────────────────────────────────────────
dpd_mes = (pdf.groupby(["monthly_reporting_period", "bucket"])
           .size().unstack(fill_value=0))

# Normalizar a %
dpd_mes_pct = dpd_mes.div(dpd_mes.sum(axis=1), axis=0) * 100

orden = ["0 – Al día", "1 – 30 DPD", "2 – 60 DPD", "3+ – Default"]
dpd_mes_pct = dpd_mes_pct.reindex(columns=[c for c in orden if c in dpd_mes_pct.columns])

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#2166ac", "#92c5de", "#f4a582", "#d6604d"]
dpd_mes_pct.plot.area(ax=ax, color=colors, linewidth=0, alpha=0.85)
ax.set_title("Composición mensual del portafolio por bucket DPD (%)", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("% del portafolio")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(loc="upper left", framealpha=0.8)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.savefig(FIG_DIR / "dpd_composicion_mensual.png", dpi=150)
plt.show()


### 4.2 ODR – Observed Default Rate mensual

In [ ]:
# ── ODR mensual: % de obs Stage 1/2 que tienen default_12m=1 ─────────────────
activos = pdf[pdf["ifrs9_stage"].isin([1, 2])].copy()

odr = (activos.groupby("monthly_reporting_period")
       .agg(
           n_obs     = ("default_12m", "count"),
           n_default = ("default_12m", "sum"),
       )
       .assign(odr_pct = lambda d: d["n_default"] / d["n_obs"] * 100)
       .reset_index())

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# ODR
axes[0].plot(odr["monthly_reporting_period"], odr["odr_pct"],
             color="#d6604d", linewidth=1.8)
axes[0].fill_between(odr["monthly_reporting_period"], odr["odr_pct"],
                     alpha=0.18, color="#d6604d")
axes[0].set_title("ODR mensual – Observed Default Rate (Stage 1+2, horizonte 12m)", fontweight="bold")
axes[0].set_ylabel("ODR (%)")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())

# Volumen de observaciones
axes[1].bar(odr["monthly_reporting_period"], odr["n_obs"],
            width=20, color="#4393c3", alpha=0.7)
axes[1].set_title("Observaciones activas por mes")
axes[1].set_ylabel("Obs.")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig(FIG_DIR / "odr_mensual.png", dpi=150)
plt.show()

print(odr[["monthly_reporting_period","n_obs","n_default","odr_pct"]]
      .rename(columns={"odr_pct":"ODR (%)"})
      .set_index("monthly_reporting_period").tail(24).to_string())


### 4.3 Curvas de vintage – tasa de default acumulada por cohorte

In [ ]:
# ── Default acumulado por vintage × edad del préstamo ────────────────────────
# Solo Stage 1+2 para no contaminar con observaciones ya en default
vintage_dpd = (activos
    .groupby(["vintage_year",
              pd.cut(activos["dpd_numerico"].fillna(0),
                     bins=[-1, 0, 1, 2, 100],
                     labels=["0","1","2","3+"])])
    .size().unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
    .assign(pct_3plus=lambda d: d["3+"] / d["total"] * 100)
    .reset_index())

# Tasa de default_12m por vintage
odr_vintage = (activos
               .groupby("vintage_year")["default_12m"]
               .agg(odr_pct=lambda x: x.mean()*100, n_obs="count")
               .reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(odr_vintage["vintage_year"].astype(str),
              odr_vintage["odr_pct"],
              color=sns.color_palette("Blues_d", len(odr_vintage)),
              edgecolor="white")
for bar, (_, row) in zip(bars, odr_vintage.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{row['odr_pct']:.2f}%", ha="center", va="bottom", fontsize=9)
ax.set_title("ODR promedio (default_12m) por vintage year", fontweight="bold")
ax.set_xlabel("Vintage")
ax.set_ylabel("ODR (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.savefig(FIG_DIR / "odr_por_vintage.png", dpi=150)
plt.show()

display(odr_vintage.style.format({"odr_pct": "{:.3f}%", "n_obs": "{:,}"}))


### 4.4 Curvas de vintage – default acumulado por loan age

In [ ]:
# ── Para cada vintage: % de préstamos únicos que defaultearon a cada loan_age ─
# Agrupamos por (vintage, loan_age) y calculamos la tasa de default promedio

# Necesitamos loan_age en el panel
cols2 = ["loan_sequence_number", "monthly_reporting_period",
         "vintage_year", "loan_age", "default_12m", "ifrs9_stage"]
try:
    pa2 = pd.read_parquet(DATA_DIR / "panel_con_default.parquet", columns=cols2)
except Exception:
    pa2 = pdf.copy()

activos2 = pa2[pa2["ifrs9_stage"].isin([1, 2])].copy()

vintage_age = (activos2
    .groupby(["vintage_year", "loan_age"])["default_12m"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"default_12m": "default_rate"}))

fig, ax = plt.subplots(figsize=(13, 5))
vintages = sorted(vintage_age["vintage_year"].unique())
cmap = plt.cm.get_cmap("tab10", len(vintages))

for i, yr in enumerate(vintages):
    sub = vintage_age[vintage_age["vintage_year"] == yr].sort_values("loan_age")
    ax.plot(sub["loan_age"], sub["default_rate"],
            label=str(yr), color=cmap(i), linewidth=1.5)

ax.set_title("Tasa de default_12m por loan age y vintage", fontweight="bold")
ax.set_xlabel("Loan age (meses)")
ax.set_ylabel("Tasa de default a 12m (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title="Vintage", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "vintage_curves_loan_age.png", dpi=150)
plt.show()


### 4.5 Distribución mensual de DPD numérico (percentiles)

In [ ]:
# ── Percentiles del DPD numérico por mes (solo obs con DPD > 0) ───────────────
dpd_pos = pdf[pdf["dpd_numerico"] > 0].copy()

dpd_stats = (dpd_pos
    .groupby("monthly_reporting_period")["dpd_numerico"]
    .agg(p25=lambda x: x.quantile(.25),
         mediana="median",
         p75=lambda x: x.quantile(.75),
         p95=lambda x: x.quantile(.95),
         n="count")
    .reset_index())

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(dpd_stats["monthly_reporting_period"],
                dpd_stats["p25"], dpd_stats["p75"],
                alpha=0.3, color="#4393c3", label="IQR (25–75)")
ax.plot(dpd_stats["monthly_reporting_period"], dpd_stats["mediana"],
        color="#2166ac", linewidth=2, label="Mediana")
ax.plot(dpd_stats["monthly_reporting_period"], dpd_stats["p95"],
        color="#d6604d", linewidth=1.2, linestyle="--", label="P95")
ax.set_title("Distribución del DPD numérico por mes (obs con DPD > 0)", fontweight="bold")
ax.set_ylabel("DPD (bucket)")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


### 4.6 Matriz de migración de stages (promedio anual)

In [ ]:
# ── Migración Stage t → Stage t+1 dentro de cada préstamo ────────────────────
stage_cols = ["loan_sequence_number", "monthly_reporting_period", "ifrs9_stage"]
sp = pdf[["loan_sequence_number","monthly_reporting_period","ifrs9_stage"]].copy()

sp = sp.sort_values(["loan_sequence_number","monthly_reporting_period"])
sp["next_stage"] = sp.groupby("loan_sequence_number")["ifrs9_stage"].shift(-1)
sp = sp.dropna(subset=["next_stage"])
sp["ifrs9_stage"]  = sp["ifrs9_stage"].astype(int)
sp["next_stage"]   = sp["next_stage"].astype(int)

trans = (sp.groupby(["ifrs9_stage","next_stage"])
         .size()
         .unstack(fill_value=0))
trans_pct = trans.div(trans.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(trans_pct, annot=True, fmt=".1f", cmap="Blues",
            linewidths=.5, ax=ax,
            xticklabels=["Stage 1","Stage 2","Stage 3"],
            yticklabels=["Stage 1","Stage 2","Stage 3"],
            cbar_kws={"label": "% transiciones"})
ax.set_title("Matriz de migración de stages (%)", fontweight="bold")
ax.set_xlabel("Stage siguiente (t+1)")
ax.set_ylabel("Stage actual (t)")
plt.tight_layout()
plt.savefig(FIG_DIR / "matriz_migracion_stages.png", dpi=150)
plt.show()


## 5. Chequeos de calidad del panel

Diagnósticos de integridad antes de proceder con feature engineering.


In [ ]:
# ── Nulos por columna en el panel con default ─────────────────────────────────
nulos = (pdf.isnull().mean() * 100).round(2).to_frame("% nulos")
nulos["dtype"] = pdf.dtypes.astype(str)
nulos["n_nulos"] = pdf.isnull().sum()
print("Columnas con valores nulos:")
display(nulos[nulos["% nulos"] > 0].sort_values("% nulos", ascending=False))


In [ ]:
# ── Duplicados (loan × mes) ───────────────────────────────────────────────────
dupes = pdf.duplicated(subset=["loan_sequence_number","monthly_reporting_period"]).sum()
print(f"Filas duplicadas (loan × mes): {dupes:,}")


In [ ]:
# ── Distribución de loan_age ──────────────────────────────────────────────────
if "loan_age" in pdf.columns:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(pdf["loan_age"].dropna(), bins=60, color="#4393c3", edgecolor="white")
    ax.set_title("Distribución del loan age (meses)", fontweight="bold")
    ax.set_xlabel("Loan age (meses)")
    ax.set_ylabel("Frecuencia")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x)}m"))
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Préstamos únicos por vintage ──────────────────────────────────────────────
unicos = (pdf.groupby("vintage_year")["loan_sequence_number"]
          .nunique()
          .reset_index()
          .rename(columns={"loan_sequence_number":"prestamos_unicos"}))
display(unicos.style.format({"prestamos_unicos": "{:,}"}))


## 6. Paso 3 – Feature engineering (`03_feature_engineering.py`)
Genera las variables predictoras y filtra las observaciones elegibles (Stage 1 y 2).


In [ ]:
import importlib.util as _iu
spec = _iu.spec_from_file_location("m03", SRC_DIR / "03_feature_engineering.py")
m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
dataset = m.main()


In [ ]:
from config import FEATURES_MODELO, TARGET, AÑOS_ENTRENAMIENTO, AÑOS_VALIDACION, AÑOS_TEST

# ── Resumen del dataset de modelado ──────────────────────────────────────────
print(f"Shape: {dataset.shape}")
print(f"\nTarget {TARGET}:")
print(dataset[TARGET].value_counts().to_string())
print(f"\nTasa de default global: {dataset[TARGET].mean():.4%}")


In [ ]:
# ── Distribuciones de features numéricas ─────────────────────────────────────
num_feats = [f for f in FEATURES_MODELO if dataset[f].dtype in ["float32","float64"]]
n = len(num_feats)
cols_g = 4
rows_g = (n + cols_g - 1) // cols_g

fig, axes = plt.subplots(rows_g, cols_g, figsize=(16, rows_g * 3))
axes = axes.flatten()

for i, feat in enumerate(num_feats):
    axes[i].hist(dataset[feat].dropna(), bins=40, color="#4393c3", edgecolor="white")
    axes[i].set_title(feat, fontsize=9, fontweight="bold")
    axes[i].tick_params(labelsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución de features numéricas (dataset de modelado)", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── % de nulos por feature ────────────────────────────────────────────────────
nulos_feat = (dataset[FEATURES_MODELO]
              .isnull().mean().mul(100).round(2)
              .sort_values(ascending=False)
              .to_frame("% nulos"))
display(nulos_feat[nulos_feat["% nulos"] > 0])


In [ ]:
# ── Correlación entre features (solo numéricas) ───────────────────────────────
feat_num = [f for f in FEATURES_MODELO if dataset[f].dtype in ["float32","float64"]]
corr = dataset[feat_num + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=.3, ax=ax, annot_kws={"size": 7},
            cbar_kws={"shrink": .8})
ax.set_title("Matriz de correlación – features numéricas + target", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlacion_features.png", dpi=150)
plt.show()


In [ ]:
# ── Tasa de default por decil de credit_score ────────────────────────────────
ds_cs = dataset.dropna(subset=["credit_score"]).copy()
ds_cs["decil_cs"] = pd.qcut(ds_cs["credit_score"], q=10, labels=False, duplicates="drop")

cs_agg = (ds_cs.groupby("decil_cs")
          .agg(tasa_default=(TARGET,"mean"),
               n=("credit_score","count"),
               cs_min=("credit_score","min"),
               cs_max=("credit_score","max"))
          .reset_index())

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(cs_agg)), cs_agg["tasa_default"]*100,
              color=sns.color_palette("Blues_d", len(cs_agg)))
ax.set_xticks(range(len(cs_agg)))
ax.set_xticklabels([f"{int(r.cs_min)}-{int(r.cs_max)}" for _,r in cs_agg.iterrows()],
                   rotation=30, fontsize=8)
ax.set_title("Tasa de default a 12m por decil de FICO score", fontweight="bold")
ax.set_ylabel("Tasa de default (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()


In [ ]:
# ── Tasa de default por decil de LTV ─────────────────────────────────────────
ds_ltv = dataset.dropna(subset=["original_ltv"]).copy()
ds_ltv["decil_ltv"] = pd.qcut(ds_ltv["original_ltv"], q=10, labels=False, duplicates="drop")

ltv_agg = (ds_ltv.groupby("decil_ltv")
           .agg(tasa_default=(TARGET,"mean"),
                n=("original_ltv","count"),
                ltv_min=("original_ltv","min"),
                ltv_max=("original_ltv","max"))
           .reset_index())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(ltv_agg)), ltv_agg["tasa_default"]*100,
       color=sns.color_palette("Reds_d", len(ltv_agg)))
ax.set_xticks(range(len(ltv_agg)))
ax.set_xticklabels([f"{int(r.ltv_min)}-{int(r.ltv_max)}" for _,r in ltv_agg.iterrows()],
                   rotation=30, fontsize=8)
ax.set_title("Tasa de default a 12m por decil de LTV original", fontweight="bold")
ax.set_ylabel("Tasa de default (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()


## 7. Paso 4 – Entrenamiento de modelos (`04_modelado.py`)
Entrena Regresión Logística, XGBoost y Random Forest con calibración isotónica.
Partición out-of-time: train 2016–2020 | val 2022–2023 | test 2024.


In [ ]:
import importlib.util as _iu
spec = _iu.spec_from_file_location("m04", SRC_DIR / "04_modelado.py")
m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
m.main()


In [ ]:
# ── Resultados de modelos ─────────────────────────────────────────────────────
from config import TABLAS_PATH
res = pd.read_csv(TABLAS_PATH / "resultados_modelos.csv")
display(res.style.highlight_max(subset=res.select_dtypes("number").columns,
                                color="#d4efdf").format("{:.4f}", na_rep="–"))


## 8. Paso 5 – Calibración Lifetime PD (`05_calibracion_lifetime.py`)
Genera la curva de PD marginal por loan age (Lifetime PD para Stage 2).


In [ ]:
import importlib.util as _iu
spec = _iu.spec_from_file_location("m05", SRC_DIR / "05_calibracion_lifetime.py")
m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
m.main()


In [ ]:
# ── Curva Lifetime PD ────────────────────────────────────────────────────────
curva = pd.read_csv(TABLAS_PATH / "curva_pd_lifetime.csv")
display(curva.head(12))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(curva.iloc[:,0], curva.iloc[:,1], color="#d6604d", linewidth=2)
axes[0].set_title("PD marginal por loan age", fontweight="bold")
axes[0].set_xlabel("Loan age (meses)")
axes[0].set_ylabel("PD marginal")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

if curva.shape[1] >= 3:
    axes[1].plot(curva.iloc[:,0], curva.iloc[:,2], color="#4393c3", linewidth=2)
    axes[1].set_title("PD Lifetime acumulada", fontweight="bold")
    axes[1].set_xlabel("Loan age (meses)")
    axes[1].set_ylabel("PD Lifetime")
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
plt.show()


## 9. Paso 6 – Validación regulatoria (`06_validacion.py`)
Métricas exigidas por IFRS 9, EBA y BCBS: AUC/Gini, KS, Brier Score, Hosmer-Lemeshow, PSI, backtesting.


In [ ]:
import importlib.util as _iu
spec = _iu.spec_from_file_location("m06", SRC_DIR / "06_validacion.py")
m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
m.main()


In [ ]:
# ── Métricas de validación ────────────────────────────────────────────────────
met = pd.read_csv(TABLAS_PATH / "validacion_metricas.csv")
display(met)

# ── Backtesting ───────────────────────────────────────────────────────────────
bkt = pd.read_csv(TABLAS_PATH / "validacion_backtesting.csv")
display(bkt)


In [ ]:
# ── Mostrar figuras de validación ────────────────────────────────────────────
from IPython.display import Image
for img_name in ["validacion_roc.png", "validacion_backtesting.png"]:
    img_path = FIG_DIR / img_name
    if img_path.exists():
        print(f"--- {img_name} ---")
        display(Image(filename=str(img_path), width=750))


## 10. Paso 7 – Explicabilidad SHAP (`07_explicabilidad.py`)
Análisis global (beeswarm, bar plot, importancia por gain) y local (waterfall por préstamo).


In [ ]:
import importlib.util as _iu
spec = _iu.spec_from_file_location("m07", SRC_DIR / "07_explicabilidad.py")
m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
m.main()


In [ ]:
# ── Importancia SHAP ──────────────────────────────────────────────────────────
shap_imp = pd.read_csv(TABLAS_PATH / "importancia_shap_xgboost.csv")
gain_imp = pd.read_csv(TABLAS_PATH / "importancia_gain_xgboost.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

top_shap = shap_imp.head(15).sort_values(shap_imp.columns[1])
axes[0].barh(top_shap.iloc[:,0], top_shap.iloc[:,1], color="#4393c3")
axes[0].set_title("Importancia SHAP (media |SHAP|) – Top 15", fontweight="bold")
axes[0].set_xlabel("mean(|SHAP value|)")

top_gain = gain_imp.head(15).sort_values(gain_imp.columns[1])
axes[1].barh(top_gain.iloc[:,0], top_gain.iloc[:,1], color="#d6604d")
axes[1].set_title("Importancia por Gain (XGBoost) – Top 15", fontweight="bold")
axes[1].set_xlabel("Gain")

plt.tight_layout()
plt.show()


In [ ]:
# ── Mostrar gráficos SHAP generados ──────────────────────────────────────────
from IPython.display import Image
for img_name in ["shap_summary_xgboost.png", "shap_barplot_xgboost.png",
                 "shap_waterfall_prestamo_503.png", "shap_waterfall_prestamo_4986.png"]:
    img_path = FIG_DIR / img_name
    if img_path.exists():
        print(f"--- {img_name} ---")
        display(Image(filename=str(img_path), width=750))


## 11. Ejecutar el pipeline completo (alternativa rápida)

> Corre las secciones 1–10 en una sola celda usando `pipeline_completo.py`.
> Útil para re-ejecución completa desde cero.


In [ ]:
# ⚠️  Esta celda re-ejecuta TODO el pipeline de cero. Descomentarla para usar.

# import importlib.util as _iu
# spec = _iu.spec_from_file_location("pipeline", SRC_DIR / "pipeline_completo.py")
# m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
# m.main()
